# portfolio-construction_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (132).ipynb`

| Field | Value |
|---|---|
| Section | `portfolio_construction` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 10 |
| Detected functions | calculate_daily_correlation |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
# Now you can load the entire rows with ICICIBANK.NSE if required
BuyPrice_Stock1 = pd.read_parquet('OCT/TickerFiles/AXISBANK_NSE.parquet')
BuyPrice_Stock1

In [ ]:
import pandas as pd
BuyPrice_Stock2 = pd.read_parquet('OCT/TickerFiles/ICICIBANK_NSE.parquet')
BuyPrice_Stock2

In [ ]:
import pandas as pd

# Assuming df1 and df2 are your two DataFrames and 'Date_Time' is the common column
combined_df = pd.merge(BuyPrice_Stock1, BuyPrice_Stock2, on='Date_Time', how='inner')  # 'inner' join will keep only the rows where 'Date_Time' is common in both datasets
combined_df.head()


In [ ]:
import pandas as pd

# Assuming combined_df is already defined and 'data' DataFrame is created
data = pd.DataFrame({
    'DateTime': combined_df['Date_Time'],
    'BuyPrice_Stock1': combined_df['BuyPrice_y'],
    'BuyPrice_Stock2': combined_df['BuyPrice_x'],
    'SellPrice_Stock1': combined_df['SellPrice_y'],
    'SellPrice_Stock2': combined_df['SellPrice_x']
})

# Remove rows where any price column has a 0 value
data = data[(data[['BuyPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock1', 'SellPrice_Stock2']] != 0).all(axis=1)]

# Display the resulting DataFrame
data


In [ ]:
# Ensure 'DateTime' is in datetime format
data['DateTime'] = pd.to_datetime(data['DateTime'])

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(data['DateTime'], data['BuyPrice_Stock1'], label='Buy Price Stock1', color='green')
plt.plot(data['DateTime'], data['BuyPrice_Stock2'], label='Buy Price Stock2', color='red')


# Formatting the plot
plt.xlabel('DateTime')
plt.ylabel('Price')
plt.title('Buy and Sell Prices of Stock1 and Stock2')
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.grid(True)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Sample data structure
# data = pd.DataFrame({
#     'datetime': [...], 
#     'BuyPrice_Stock1': [...], 
#     'SellPrice_Stock1': [...], 
#     'BuyPrice_Stock2': [...], 
#     'SellPrice_Stock2': [...]
# })

# Ensure the 'datetime' column is in datetime format
data['datetime'] = pd.to_datetime(data['DateTime'])

# Extract the date directly for grouping
data['date'] = data['datetime'].dt.date

# Function to calculate daily correlations for a specific price type
def calculate_daily_correlation(df, price_type1, price_type2):
    return (
        df.groupby('date')[[price_type1, price_type2]]
        .corr()
        .iloc[0::2, -1]
        .reset_index(level=1, drop=True)
    )

# Calculate correlations for BuyPrice and SellPrice
daily_corr_buy = calculate_daily_correlation(data, 'BuyPrice_Stock1', 'BuyPrice_Stock2')

# Combine results into a single DataFrame
daily_correlation = pd.DataFrame({
    'date': daily_corr_buy.index,
    'BuyPrice_correlation': daily_corr_buy.values,
}).reset_index(drop=True)

print(daily_correlation)


In [ ]:
import pandas as pd
import numpy as np

# Load your dataset
data['datetime'] = pd.to_datetime(data['DateTime'])
data.set_index('datetime', inplace=True)  # Set datetime as index

# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_window = 43  # Moving average window
z_score_threshold = 0.2  # Z-score threshold

# Initialize variables
portfolio_value = initial_capital
current_position = 0
entry_price1, entry_price2 = None, None
entry_datetime = None
cumulative_pnl = 0
tradebook = []

# Loop through the data and execute trades
for i, row in data.iterrows():
    # Z-score and trade execution logic
    z_score = row['z_score']
    sell_price1, buy_price1 = row['SellPrice_Stock1'], row['BuyPrice_Stock1']
    sell_price2, buy_price2 = row['SellPrice_Stock2'], row['BuyPrice_Stock2']
    current_datetime = row.name  # Use index as the datetime

    # Skip NaN prices
    if pd.isna(sell_price1) or pd.isna(buy_price2):
        continue

    # Position size calculations based on portfolio value
    max_risk = portfolio_value * risk_percentage
    position_size1_buy = (max_risk / (2 * buy_price1)).round()
    position_size1_sell = (max_risk / (2 * sell_price1)).round()
    position_size2_buy = (max_risk / (2 * buy_price2)).round()
    position_size2_sell = (max_risk / (2 * sell_price2)).round()

    # Entry and exit logic
    if z_score > z_score_threshold and current_position == 0:  # Short stock1, long stock2
        entry_price1, entry_price2 = sell_price1, buy_price2
        entry_datetime = current_datetime
        current_position = -1

    elif z_score < -z_score_threshold and current_position == 0:  # Long stock1, short stock2
        entry_price1, entry_price2 = buy_price1, sell_price2
        entry_datetime = current_datetime
        current_position = 1

    elif current_position != 0 and (abs(z_score) < 0.01 or (current_datetime.hour == 15 and current_datetime.minute == 28)):
        # Calculate PnL
        position_size1 = position_size1_sell if current_position == -1 else position_size1_buy
        position_size2 = position_size2_buy if current_position == -1 else position_size2_sell
        exit_price1 = buy_price1 if current_position == -1 else sell_price1
        exit_price2 = sell_price2 if current_position == -1 else buy_price2

        pnl = (position_size1 * (exit_price1 - entry_price1)) + (position_size2 * (entry_price2 - exit_price2))
        transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) +
                                            position_size2 * (entry_price2 + exit_price2))
        pnl -= transaction_cost
        cumulative_pnl += pnl
        portfolio_value += pnl  # Update portfolio value after trade

        tradebook.append({
            'Entry Price Stock1': entry_price1,
            'Entry Price Stock2': entry_price2,
            'Exit Price Stock1': exit_price1,
            'Exit Price Stock2': exit_price2,
            'Trade Type': 'Short' if current_position == -1 else 'Long',
            'Position Size Stock1': position_size1,
            'Position Size Stock2': position_size2,
            'PnL': round(pnl, 2),
            'Transaction Cost': round(transaction_cost, 2),
            'Portfolio Value': round(portfolio_value, 2),
            'Cumulative PnL': round(cumulative_pnl, 2),
            'Entry Date': entry_datetime.date(),
            'Entry Time': entry_datetime.time(),
            'Exit Date': current_datetime.date(),
            'Exit Time': current_datetime.time(),
            'Time Difference (minutes)': (current_datetime - entry_datetime).total_seconds() / 60
        })

        current_position = 0  # Reset position

# Convert tradebook to DataFrame and save
tradebook_df = pd.DataFrame(tradebook)
tradebook_df.to_csv('tradebook.csv', index=False)

print(tradebook_df)


In [ ]:
import pandas as pd
import numpy as np

# Load your dataset (replace this with your actual data loading method)
# Ensure your DataFrame has the columns: 'datetime', 'BuyPrice_Stock1', 'SellPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock2'
data['datetime'] = pd.to_datetime(data['DateTime'])

# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_windows = [750, 1500, 2250, 3000, 4500, 6000, 7500]  # Moving average windows
z_score_thresholds = [0.2, 0.25, 0.3]  # Z-score thresholds

# Results storage
results = []

# Loop through each combination of MA and Z-score threshold
for ma_window in ma_windows:
    # Calculate rolling statistics for Z-score
    data['spread'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) - (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
    data['rolling_mean'] = data['spread'].rolling(window=ma_window).mean()
    data['rolling_std'] = data['spread'].rolling(window=ma_window).std()
    data['z_score'] = (data['spread'] - data['rolling_mean']) / data['rolling_std']

    # Precompute position sizes
    data['max_risk'] = initial_capital * risk_percentage
    for col, price_col in zip(['position_size1_buy', 'position_size1_sell', 'position_size2_buy', 'position_size2_sell'], 
                              ['BuyPrice_Stock1', 'SellPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock2']):
        data[col] = (data['max_risk'] / (2 * data[price_col])).round()

    # Iterate through Z-score thresholds
    for z_threshold in z_score_thresholds:
        # Initialize variables for backtesting
        portfolio_value = initial_capital
        peak_portfolio_value = initial_capital
        max_drawdown = 0
        current_position = 0
        cumulative_pnl = 0
        tradebook = []

        # Iterate through the dataset row-wise
        for _, row in data.iterrows():
            z_score = row['z_score']
            sell_price1, buy_price1 = row['SellPrice_Stock1'], row['BuyPrice_Stock1']
            sell_price2, buy_price2 = row['SellPrice_Stock2'], row['BuyPrice_Stock2']
            current_datetime = row['datetime']

            # Skip iteration if prices are NaN
            if pd.isna([sell_price1, buy_price1, sell_price2, buy_price2]).any():
                continue

            # Entry logic
            if z_score > z_threshold and current_position == 0:  # Short stock1, long stock2
                entry_price1, entry_price2 = sell_price1, buy_price2
                entry_datetime = current_datetime
                current_position = -1

            elif z_score < -z_threshold and current_position == 0:  # Long stock1, short stock2
                entry_price1, entry_price2 = buy_price1, sell_price2
                entry_datetime = current_datetime
                current_position = 1

            # Exit logic
            elif current_position != 0 and (abs(z_score) < 0.01 or (current_datetime.hour == 15 and current_datetime.minute >= 30)):
                # Determine position sizes and exit prices
                position_size1 = row['position_size1_sell'] if current_position == -1 else row['position_size1_buy']
                position_size2 = row['position_size2_buy'] if current_position == -1 else row['position_size2_sell']
                exit_price1 = buy_price1 if current_position == -1 else sell_price1
                exit_price2 = sell_price2 if current_position == -1 else buy_price2

                # Calculate PnL and transaction costs
                pnl = (position_size1 * (exit_price1 - entry_price1)) + (position_size2 * (entry_price2 - exit_price2))
                transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) +
                                                    position_size2 * (entry_price2 + exit_price2))
                pnl -= transaction_cost
                cumulative_pnl += pnl
                portfolio_value += pnl

                # Update drawdown
                peak_portfolio_value = max(peak_portfolio_value, portfolio_value)
                drawdown = (peak_portfolio_value - portfolio_value) / peak_portfolio_value
                max_drawdown = max(max_drawdown, drawdown)

                # Record trade
                tradebook.append({
                    'Entry DateTime': entry_datetime,
                    'Exit DateTime': current_datetime,
                    'Position Size Stock1': position_size1,
                    'Position Size Stock2': position_size2,
                    'PnL': pnl,
                    'Portfolio Value': portfolio_value,
                    'Transaction Cost': transaction_cost
                })

                # Reset position
                current_position = 0

        # Store results for the current configuration
        results.append({
            'Moving Average Window': ma_window,
            'Z-score Threshold': z_threshold,
            'Total PnL': cumulative_pnl,
            'Total Trades': len(tradebook),
            'Final Portfolio Value': portfolio_value,
            'Max Drawdown (%)': round(max_drawdown * 100, 2),
        })

# Convert results to DataFrame and export
results_df = pd.DataFrame(results)
results_df.to_csv('robustness_test_results.csv', index=False)

# Display results
print(results_df)
